# From Function Calling to the Agent Loop

The runnable code of [From Function Calling to the Agent Loop](https://residual-stream.lvtan71.workers.dev/posts/function-calling-to-agent-loop/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/function-calling-to-agent-loop.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 19 min end to end.
Downloads Qwen2.5-1.5B-Instruct and Qwen2.5-3B-Instruct (about 9 GB together) and MiniLM.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "function-calling-to-agent-loop"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/function-calling-to-agent-loop/agent.py', 'data/questions.json', 'data/blog-a7da2a2.tar.gz']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Function calling is text plus a protocol

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/function-calling-to-agent-loop/#function-calling-is-text-plus-a-protocol)

In [ ]:
import json
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download

# What function calling looks like as text: one exchange rendered through
# Qwen2.5's chat template. The tool's schema goes into the system prompt;
# the model's call is JSON between <tool_call> tags; the application's
# result goes back as a tool message, which the template wraps in
# <tool_response> tags. (Search result abridged.)
tok = AutoTokenizer.from_pretrained(snapshot_download("Qwen/Qwen2.5-1.5B-Instruct",
                                                      revision="989aa7980e4cf806f80c7fef2b1adb7bc71aa306"))
tools = [{"type": "function", "function": {"name": "search_posts", "description": "Search the blog.",
          "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}]
call = {"name": "search_posts", "arguments": {"query": "online softmax running maximum"}}
result = [{"post": "flashattention-explained", "heading": "The online softmax", "snippet": "Keep a running maximum m..."}]
messages = [{"role": "system", "content": "You answer questions about a technical blog, using its tools."},
            {"role": "user", "content": "How can a maximum and a running sum be updated incrementally?"},
            {"role": "assistant", "content": f"<tool_call>\n{json.dumps(call)}\n</tool_call>"},
            {"role": "tool", "content": json.dumps(result)}]
print(tok.apply_chat_template(messages, tools=tools, add_generation_prompt=True, tokenize=False))

The post printed:

```text
<|im_start|>system
You answer questions about a technical blog, using its tools.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "search_posts", "description": "Search the blog.", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|im_end|>
<|im_start|>user
How can a maximum and a running sum be updated incrementally?<|im_end|>
<|im_start|>assistant
<tool_call>
{"name": "search_posts", "arguments": {"query": "online softmax running maximum"}}
</tool_call><|im_end|>
<|im_start|>user
<tool_response>
[{"post": "flashattention-explained", "heading": "The online softmax", "snippet": "Keep a running maximum m..."}]
</tool_response><|im_end|>
<|im_start|>assistant
```

## The transcript is the memory, and it grows

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/function-calling-to-agent-loop/#the-transcript-is-the-memory-and-it-grows)

In [ ]:
# What an agent's transcript costs. Every turn resends everything so far: a
# P-token prefix (system prompt and tools), then g new tokens per turn (the
# model's call plus the tool's result). Turn t reads P + (t - 1) g tokens of
# context, so T turns read T P + g T (T - 1) / 2 in all: quadratic in T.
# With prompt caching ("Prompt Caching"), turn 1 writes the prefix to the cache at
# 1.25x the base input price; each later turn reads everything sent before
# at 0.1x and writes the previous turn's new tokens at 1.25x.
P, g, price = 5_000, 1_500, 3 / 1e6           # tokens; dollars per input token (an illustrative $3 per million)
print(f"prefix {P:,} tokens, {g:,} new per turn, ${price * 1e6:.0f} per million input tokens")
print(f"{'turns':>6s} {'final context':>14s} {'tokens read in all':>19s} {'uncached':>9s} {'cached':>8s}")
for T in (5, 10, 20, 50):
    total = T * P + g * T * (T - 1) // 2
    cached = P * 1.25 + sum((P + (t - 2) * g) * 0.1 + g * 1.25 for t in range(2, T + 1))   # reread, plus the new turn written
    print(f"{T:6d} {P + (T - 1) * g:12,d}   {total:17,d}   ${total * price:7.2f}  ${cached * price:6.2f}")

The post printed:

```text
prefix 5,000 tokens, 1,500 new per turn, $3 per million input tokens
 turns  final context  tokens read in all  uncached   cached
     5       11,000              40,000   $   0.12  $  0.05
    10       18,500             117,500   $   0.35  $  0.10
    20       33,500             385,000   $   1.16  $  0.23
    50       78,500           2,087,500   $   6.26  $  0.90
```

## An experiment: an agent on this blog

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/function-calling-to-agent-loop/#an-experiment-an-agent-on-this-blog)

In [ ]:
%%writefile agent.py
import collections
import glob
import json
import math
import os
import re
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

# An agent on this blog: Qwen2.5 (1.5B or 3B) in a loop with two tools,
# search_posts and read_section, until it answers or runs out of turns.
# The model never runs anything: it writes a tool call as JSON, the loop
# runs it and appends the result, and the model is called again.
# Compared with one retrieval and one answer, on 30 questions each answered
# by one known section (questions.json). Scored three ways: did a search
# return the right post, does the answer cite any source, and does it cite
# the right post. Posts that discuss these questions are left out.

# --- search: the hybrid of "Semantic and Hybrid Search" (BM25 and MiniLM, fused by rank) ---
EXCLUDE = {"function-calling-to-agent-loop", "agentic-graph-and-vectorless-rag", "recursive-language-models"}
SKIP = {"table of contents", "takeaways", "references", "where this leads"}
secs = []
for path in sorted(glob.glob("content/posts/*.mdx")):
    slug = os.path.basename(path)[:-4]
    if slug in EXCLUDE:
        continue
    text = re.sub(r"```.*?```|\$\$.*?\$\$", " ", open(path).read().split("---", 2)[-1], flags=re.S)
    parts = re.split(r"^## (.+)$", re.sub(r"<[^>]+>|^import .*$", " ", text, flags=re.M), flags=re.M)
    for h, body in zip(parts[1::2], parts[2::2]):
        h = re.sub(r"[*_`]", "", h).strip()
        if h.lower() not in SKIP and len(body.strip()) >= 200:
            secs.append({"post": slug, "heading": h, "body": " ".join(body.split())})
words = lambda t: re.findall(r"\w+", t.lower())
bags = [collections.Counter(words(s["heading"] + " " + s["body"])) for s in secs]
lens = np.array([sum(b.values()) for b in bags]); df = collections.Counter(w for b in bags for w in b)
def bm25(q, k1=1.5, b=0.75):
    s = np.zeros(len(secs))
    for w in set(words(q)) & df.keys():
        tf = np.array([d[w] for d in bags]); idf = math.log((len(secs) - df[w] + 0.5) / (df[w] + 0.5) + 1)
        s += idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * lens / lens.mean()))
    return s
st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
owner, passages = zip(*[(i, " ".join(w[j:j + 80])) for i, s in enumerate(secs)
                        for w in [s["body"].split()] for j in range(0, len(w), 80)])
E = st.encode(list(passages), batch_size=64, normalize_embeddings=True); owner = np.array(owner)
def dense(q):
    best = np.full(len(secs), -np.inf); np.maximum.at(best, owner, E @ st.encode([q], normalize_embeddings=True)[0])
    return best
def hybrid(q, k=60):
    return sum(1 / (k + np.argsort(np.argsort(-f(q))) + 1) for f in (bm25, dense))

# --- the tools ---
SEEN = set()                                        # posts any search returned, for scoring
def search_posts(query):
    top = [{"post": secs[i]["post"], "heading": secs[i]["heading"], "snippet": " ".join(secs[i]["body"].split()[:60])}
           for i in np.argsort(-hybrid(query))[:3]]
    SEEN.update(t["post"] for t in top)
    return top
def read_section(post, heading):
    for s in secs:
        if s["post"] == post and s["heading"].lower() == heading.lower():
            return " ".join(s["body"].split()[:400])
    return "error: no such section; use a post and heading exactly as search_posts returned them"
RUN = {"search_posts": search_posts, "read_section": read_section}
TOOLS = [
    {"type": "function", "function": {"name": "search_posts", "description": "Search the blog. Returns the 3 best "
     "matching sections: post, heading, and the start of the text.",
     "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "read_section", "description": "Read one section in full.",
     "parameters": {"type": "object", "properties": {"post": {"type": "string"}, "heading": {"type": "string"}},
                    "required": ["post", "heading"]}}}]
SYSTEM = ("You answer questions about a technical blog, using its tools. When you know the answer, reply "
          "without a tool call: the answer in one or two sentences, then 'Source: <post>'.")
STRICT = SYSTEM + (" Always call search_posts before answering, never answer from memory, and give as the "
                   "source the post name a tool returned.")

# --- the model and the loop ---
import sys
from huggingface_hub import snapshot_download
GPU = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"  # notebook: the post ran on "mps"
PATH = {"1.5B": snapshot_download("Qwen/Qwen2.5-1.5B-Instruct",
                                  revision="989aa7980e4cf806f80c7fef2b1adb7bc71aa306"),
        "3B": snapshot_download("Qwen/Qwen2.5-3B-Instruct",
                                revision="aa8e72537993ba99e69dfaafa59ed015b17504d1")}[sys.argv[1] if len(sys.argv) > 1 else "1.5B"]
tok = AutoTokenizer.from_pretrained(PATH)
lm = AutoModelForCausalLM.from_pretrained(PATH, dtype=torch.float16).to(GPU).eval()
@torch.no_grad()
def generate(messages, tools=TOOLS, force_call=False):
    """force_call starts the reply with "<tool_call>", approximating an API's
    tool_choice = "required": the model must start a tool call."""
    prompt = tok.apply_chat_template(messages, tools=tools, add_generation_prompt=True, tokenize=False)
    prefix = "<tool_call>\n" if force_call else ""
    x = tok(prompt + prefix, return_tensors="pt").input_ids.to(GPU)
    out = lm.generate(x, max_new_tokens=200, do_sample=False, temperature=None, top_p=None, top_k=None)
    return prefix + tok.decode(out[0, x.shape[1]:], skip_special_tokens=True), x.shape[1]

def agent(question, system=SYSTEM, force_first=False, max_turns=6):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": question}]
    read, bad, calls_made = [], 0, 0
    for turn in range(1, max_turns + 1):
        text, n = generate(messages, force_call=force_first and turn == 1); read.append(n)
        calls = re.findall(r"<tool_call>\s*(\{.*?)\s*</tool_call>", text, flags=re.S)
        if not calls:                                   # no call: the model says it's done
            return text, read, bad, calls_made
        messages.append({"role": "assistant", "content": text})
        for c in calls:                                 # the loop runs the tools, not the model
            try:
                call = json.JSONDecoder().raw_decode(c)[0]; result = RUN[call["name"]](**call["arguments"])
            except Exception as e:
                bad += 1; result = f"error: {type(e).__name__}: {e}"
            calls_made += 1
            messages.append({"role": "tool", "content": json.dumps(result)})
    return "(ran out of turns)", read, bad, calls_made

def one_shot(question):
    """No loop: retrieve once with the question itself, answer once."""
    text, n = generate([{"role": "system", "content": SYSTEM}, {"role": "user", "content":
                        f"Search results: {json.dumps(search_posts(question))}\n\nQuestion: {question}"}], tools=None)
    return text, [n], 0, 1

items = json.load(open("questions.json"))
print(f"{len(secs)} sections; {len(items)} questions; Qwen2.5-{sys.argv[1] if len(sys.argv) > 1 else '1.5B'}-Instruct, greedy\n")
print(f"{'':32s} {'found':>6s} {'cites any':>10s} {'cites right':>12s} {'model calls':>12s} {'tool calls':>11s} "
      f"{'last context':>13s} {'tokens read':>12s} {'no tool':>8s} {'malformed':>10s}")
for name, fn in (("one retrieval, one answer", one_shot), ("agent loop", agent),
                 ("agent, 'always search first'", lambda q: agent(q, STRICT)),
                 ("agent, first call forced", lambda q: agent(q, STRICT, force_first=True))):
    ok, found, cites, ncalls, ntools, last, total, none, bad = [], [], [], [], [], [], [], 0, 0
    for it in items:
        SEEN.clear()
        text, read, b, tools_used = fn(it["question"])
        found.append(it["post"] in SEEN); cites.append("Source:" in text)
        with open(f"agent_log_{sys.argv[1] if len(sys.argv) > 1 else '1.5B'}.jsonl", "a") as f:        # every answer, for checking by hand
            f.write(json.dumps({"method": name, "post": it["post"], "answer": text, "calls": len(read), "found": it["post"] in SEEN,
                                "tools": tools_used, "malformed": b}) + "\n")
        ok.append(it["post"] in text.split("Source:")[-1]); ncalls.append(len(read)); ntools.append(tools_used)
        last.append(read[-1]); total.append(sum(read)); bad += b; none += fn is not one_shot and tools_used == 0
    print(f"{name:32s} {sum(found):>6d} {sum(cites):>10d} {sum(ok):>12d} {np.mean(ncalls):>12.1f} {np.mean(ntools):>11.1f} "
          f"{np.mean(last):>13,.0f} {np.mean(total):>12,.0f} {none:>8d} {bad:>10d}", flush=True)

In [ ]:
# The post ran the loop with both model sizes:
%run agent.py 1.5B
%run agent.py 3B

The post printed:

```text
446 sections; 30 questions; Qwen2.5-1.5B-Instruct, greedy

                                  found  cites any  cites right  model calls  tool calls  last context  tokens read  no tool  malformed
one retrieval, one answer            23         28           17          1.0         1.0           427          427        0          0
agent loop                            0         26            0          1.0         0.0           286          286       30          0
agent, 'always search first'          0         25            0          1.0         0.0           309          309       30          0
agent, first call forced             18          1            2          2.0         1.0           710        1,021        0          0
446 sections; 30 questions; Qwen2.5-3B-Instruct, greedy

                                  found  cites any  cites right  model calls  tool calls  last context  tokens read  no tool  malformed
one retrieval, one answer            23         30           23          1.0         1.0           427          427        0          0
agent loop                            4         29            3          1.2         0.2           371          428       24          0
agent, 'always search first'          9         28            3          1.4         0.4           491          624       17          0
agent, first call forced             21         24           10          2.0         1.0           720        1,030        0          0
```

In [ ]:
import glob
import json
import os
import re

# Rescoring the logged answers. agent.py counted an answer as citing the
# right post if the post's name appeared anywhere after "Source:", and
# anywhere at all when there was no "Source:" line; a list of several
# posts also passed. Here a citation counts only if the answer has a
# "Source:" line naming exactly one of the blog's posts, and that post is
# the right one. Answers are also sorted by what their source line holds.
SLUGS = {os.path.basename(p)[:-4] for p in glob.glob("content/posts/*.mdx")}
def cited(answer):
    if "Source:" not in answer:
        return "no source line", None
    line = answer.split("Source:")[-1]
    named = {s for s in SLUGS if s in line}
    if len(named) == 1:
        return "one real post", named.pop()
    if len(named) > 1:
        return "several posts", None
    return "placeholder or invented", None

for model in ("1.5B", "3B"):
    rows = [json.loads(l) for l in open(f"agent_log_{model}.jsonl")]
    print(f"Qwen2.5-{model}-Instruct")
    print(f"  {'':32s} {'found':>6s} {'cites right':>12s}   source line: none / one real post / several / placeholder or invented")
    for method in dict.fromkeys(r["method"] for r in rows):
        rs = [r for r in rows if r["method"] == method]
        kinds = [cited(r["answer"]) for r in rs]
        right = sum(slug == r["post"] for (_, slug), r in zip(kinds, rs))
        count = lambda k: sum(kk == k for kk, _ in kinds)
        print(f"  {method:32s} {sum(r['found'] for r in rs):>6d} {right:>12d}   "
              f"{count('no source line'):>4d} / {count('one real post'):>2d} / {count('several posts'):>2d} / "
              f"{count('placeholder or invented'):>2d}")

The post printed:

```text
Qwen2.5-1.5B-Instruct
                                    found  cites right   source line: none / one real post / several / placeholder or invented
  one retrieval, one answer            23           17      2 / 26 /  0 /  2
  agent loop                            0            0      4 /  0 /  0 / 26
  agent, 'always search first'          0            0      5 /  0 /  0 / 25
  agent, first call forced             18            0     29 /  0 /  1 /  0
Qwen2.5-3B-Instruct
                                    found  cites right   source line: none / one real post / several / placeholder or invented
  one retrieval, one answer            23           23      0 / 30 /  0 /  0
  agent loop                            4            3      1 /  4 /  0 / 25
  agent, 'always search first'          9            3      2 /  3 /  1 / 24
  agent, first call forced             21           10      6 / 15 /  1 /  8
```